# Experimental LR calibration comparison
This notebook reads the completed R3-N1 LR export and temperature fit. All outputs stay under experiments/r3_n1/lr_calibration. It does not train the LR model, access test data, change locks, or promote a calibrator. The operating role was previously inspected; results are exploratory.
Sync this entire experiment folder, src/, and runs/r3/r3_n1_lr/selected_lr_v1/ to Drive before running. Select a fresh run ID. Dependencies numpy, scipy, pyarrow and pytest must already be installed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project').resolve()  # Change if needed
EXP = ROOT / 'experiments/r3_n1/lr_calibration'
RUN_ID = 'colab_' + __import__('datetime').datetime.now(__import__('datetime').timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN = EXP / 'runs' / RUN_ID
required = [EXP/'scripts/run.py', EXP/'scripts/analyze.py', EXP/'reports/plan.md', EXP/'tests/test_calibration.py', ROOT/'runs/r3/r3_n1_lr/selected_lr_v1/export/evidence.json', ROOT/'runs/r3/r3_n1_lr/selected_lr_v1/fit/temperature.json']
missing = [str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('Missing files: ' + ', '.join(missing))
if RUN.exists(): raise FileExistsError(RUN)
print('Project:', ROOT, '\nOutput:', RUN)

In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT/'src')
subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', str(EXP/'tests')], cwd=str(ROOT), env=env, check=True)

In [ ]:
subprocess.run([sys.executable, str(EXP/'scripts/run.py'), '--output', str(RUN)], cwd=str(ROOT), env=env, check=True)

In [ ]:
subprocess.run([sys.executable, str(EXP/'scripts/analyze.py'), '--run', str(RUN)], cwd=str(ROOT), env=env, check=True)
from IPython.display import SVG, display
display(SVG(filename=str(RUN/'analysis/operating_reliability.svg')))
print((RUN/'analysis/threshold_comparison.csv').read_text())

Each run writes to a fresh directory and will not overwrite a prior run. Add reviewed runs to the experiment ledger manually. The analysis currently expects Platt to be selected and stops if another method wins; inspect its run evidence if that occurs.